TITANIC DATASET

In [ ]:
import pandas as pd
import numpy as np
train="https://raw.githubusercontent.com/abysswalkervoid/Titanic---Machine-Learning-from-Disaster/refs/heads/main/train.csv"
data = pd.read_csv(train)

test="https://raw.githubusercontent.com/abysswalkervoid/Titanic---Machine-Learning-from-Disaster/refs/heads/main/test.csv"
test = pd.read_csv(test)

PassengerID=test["PassengerId"]
# data.head()
#print(data.columns)

In [ ]:
data.head(100)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,0,3,"Shorney, Mr. Charles Joseph",male,NaN,0,0,374910,8.0500,NaN,S
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C
97,98,1,1,"Greenfield, Mr. William Bertram",male,23.0,0,1,PC 17759,63.3583,D10 D12,C
98,99,1,2,"Doling, Mrs. John T (Ada Julia Bone)",female,34.0,0,1,231919,23.0000,NaN,S


Column data
	PassengerId	Survived	Pclass	Name	Sex	Age	SibSp	Parch	Ticket	Fare	Cabin	Embarked



1. passenger id - no use ❌
2. survived - target class ✅
3. pclass - 1/2/3 economic status ✅
4. Name - mr/mrs/miss ✅->0,1,2,3,4
5. sex - useful ✅->0,1
6. Age - useful ✅ ->category->0,1,2,3,4
7. sibsp - sibling/spouses - useful - ✅
8. parch - parent/children - useful - ✅
9. ticket - no use - ❌
10. fare - useful - ✅
11. cabin - not useful - has lot of missing val - ❌
12. embarked - tells where the ship landed,, - useful - ✅

create new column->sibsp and parch->family size
using this->is alone?->0,1


In [ ]:
def data_cleaning(a):
  a=a.drop(["PassengerId","Ticket","Cabin"],axis=1)   #remove waste column

  cols_with_missing_values=["SibSp","Parch","Fare","Age"]
  for i in cols_with_missing_values:
    a[i].fillna(a[i].median(),inplace=True)           #missing values ->median
  a["Embarked"].fillna("U",inplace=True)
  a["Title"]=a["Name"].str.extract('([A-Za-z]+)\\.',expand=False)  #take required info
  a=a.drop(["Name"],axis=1)
  return a

data=data_cleaning(data)
test=data_cleaning(test)

'''too many titles found using label encoder,,->convert them
['Capt' 'Col' 'Countess' 'Don' 'Dona' 'Dr' 'Jonkheer' 'Lady'
 'Major' 'Master' 'Miss' 'Mlle' 'Mme' 'Mr' 'Mrs' 'Ms' 'Rev' 'Sir']
to ['Master' 'Miss' 'Mr' 'Mrs' 'Rare'] '''


for dataset in [data, test]:
    dataset["Title"] = dataset["Title"].replace(["Mlle", "Ms"], "Miss")
    dataset["Title"] = dataset["Title"].replace("Mme", "Mrs")
    dataset["Title"] = dataset["Title"].replace(
        ["Countess", "Lady", "Sir", "Jonkheer", "Don", "Dona", "Dr", "Rev", "Major", "Col", "Capt"],
        "Rare"
    )


from sklearn.preprocessing import LabelEncoder      #male,female -> 0,1 convert,,
for x in ["Sex","Embarked","Title"]:
  lbl=LabelEncoder()
  lbl.fit(pd.concat([data[x],test[x]]))
  data[x]=lbl.transform(data[x])
  test[x]=lbl.transform(test[x])
  print(lbl.classes_)


data["FamilySize"] = data["SibSp"] + data["Parch"] + 1  #making a new column
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

data["AgeGroup"] = pd.cut(data["Age"], bins=[0,12,18,30,50,80], labels=[0,1,2,3,4])   #1-100->0,1,2,3,4
test["AgeGroup"] = pd.cut(test["Age"], bins=[0,12,18,30,50,80], labels=[0,1,2,3,4])

data["AgeGroup"] = data["AgeGroup"].astype(int)
test["AgeGroup"] = test["AgeGroup"].astype(int)

/tmp/ipykernel_391/967910821.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  a[i].fillna(a[i].median(),inplace=True)           #missing values ->median
/tmp/ipykernel_391/967910821.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value

['female' 'male']
['C' 'Q' 'S' 'U']
['Master' 'Miss' 'Mr' 'Mrs' 'Rare']


In [ ]:
#data.head(100)
data.columns
#age/age group ->use the required based on the model


Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked', 'Title', 'FamilySize', 'IsAlone', 'AgeGroup'],
      dtype='object')

Model Building

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y=data["Survived"]
x=data.drop(["Survived"],axis=1)

x_train,x_val,y_train,y_val=train_test_split(x,y,test_size=0.2,random_state=128)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
test_scaled = scaler.transform(test)

logistic regression


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

classifier = LogisticRegression(random_state=0,max_iter=1000).fit(x_train,y_train)
predictions=classifier.predict(x_val)
print(accuracy_score(y_val,predictions))

0.8100558659217877


logistic regression with scaled

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

classifier = LogisticRegression(random_state=0, max_iter=2000).fit(x_train_scaled, y_train)
predictions = classifier.predict(x_val_scaled)
print(accuracy_score(y_val, predictions))


0.8100558659217877


Linear regression with c


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

for c in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(C=c, random_state=0, max_iter=2000)
    model.fit(x_train_scaled, y_train)
    preds = model.predict(x_val_scaled)
    print("C =", c, "Accuracy:", accuracy_score(y_val, preds))

C = 0.01 Accuracy: 0.8435754189944135
C = 0.1 Accuracy: 0.8100558659217877
C = 1 Accuracy: 0.8100558659217877
C = 10 Accuracy: 0.8044692737430168
C = 100 Accuracy: 0.8044692737430168


linear regression with GridSearchCV

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid = GridSearchCV(LogisticRegression(random_state=0, max_iter=2000), param_grid, cv=5)
grid.fit(x_train_scaled, y_train)

print("Best Params:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)


Best Params: {'C': 10, 'solver': 'liblinear'}
Best Accuracy: 0.8005909583374372


AdaBoostClassifier

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score

ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=0)
ada.fit(x_train_scaled, y_train)

ada_preds = ada.predict(x_val_scaled)
print("AdaBoost Accuracy:", accuracy_score(y_val, ada_preds))


AdaBoost Accuracy: 0.8379888268156425


GradientBoostingClassifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=0)
gb.fit(x_train_scaled, y_train)

gb_preds = gb.predict(x_val_scaled)
print("GradientBoosting Accuracy:", accuracy_score(y_val, gb_preds))


GradientBoosting Accuracy: 0.8268156424581006


XGBClassifier

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    random_state=0,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

xgb.fit(x_train_scaled, y_train)
xgb_preds = xgb.predict(x_val_scaled)

print("XGBoost Accuracy:", accuracy_score(y_val, xgb_preds))


XGBoost Accuracy: 0.8268156424581006


RandomForestClassifier


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(x_train, y_train)
rf_preds = rf.predict(x_val)
print("Random Forest Accuracy:", accuracy_score(y_val, rf_preds))
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, None],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid=params, cv=5, scoring="accuracy")
grid.fit(x_train, y_train)
print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)



Random Forest Accuracy: 0.8100558659217877
Best Parameters: {'max_depth': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best Accuracy: 0.8244262779474048


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=200,       # number of trees 🌲
    max_depth=6,            # how deep each tree can go
    random_state=0
)

rf.fit(x_train_scaled, y_train)
rf_preds = rf.predict(x_val_scaled)

print("🌲 Random Forest Accuracy:", accuracy_score(y_val, rf_preds))


🌲 Random Forest Accuracy: 0.8603351955307262


In [ ]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=0
)

lgb.fit(x_train_scaled, y_train)
lgb_preds = lgb.predict(x_val_scaled)

print("⚡ LightGBM Accuracy:", accuracy_score(y_val, lgb_preds))


[LightGBM] [Info] Number of positive: 282, number of negative: 430
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000404 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 225
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.396067 -> initscore=-0.421878
[LightGBM] [Info] Start training from score -0.421878
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=0,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('lgb', lgb),
        ('xgb', xgb)
    ],
    voting='soft'   # soft = uses probability averages 🤓
)

ensemble.fit(x_train_scaled, y_train)
ens_preds = ensemble.predict(x_val_scaled)

print("🧩 Ensemble Accuracy:", accuracy_score(y_val, ens_preds))


[LightGBM] [Info] Number of positive: 282, number of negative: 430
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000135 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 225
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.396067 -> initscore=-0.421878
[LightGBM] [Info] Start training from score -0.421878
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


GradientBoostingClassifier

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb.fit(x_train, y_train)
gb_preds = gb.predict(x_val)
print("Gradient Boosting Accuracy:", accuracy_score(y_val, gb_preds))

Gradient Boosting Accuracy: 0.8268156424581006


In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(random_state=42, n_estimators=300, learning_rate=0.05)
model.fit(x_train, y_train)
preds = model.predict(x_val)
print("XGBoost Accuracy:", accuracy_score(y_val, preds))


XGBoost Accuracy: 0.8379888268156425


Submission


In [ ]:
submissions_prediction = classifier.predict(test_scaled)
df = pd.DataFrame({"PassengerID": PassengerID.values, "Survived": submissions_prediction})
df.to_csv("gender_submission.csv", index=False)

In [ ]:
# Generate predictions on the test set using the trained logistic regression model
test_predictions = rf.predict(test_scaled)

# Create the submission DataFrame
submission_df = pd.DataFrame({'PassengerId': PassengerID, 'Survived': test_predictions})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")

Submission file created successfully!
